In [ ]:
import pandas as pd
import numpy as np

# --- Carregando todos os componentes necessários para a função ---

# DataFrame de artigos
df_artigos = pd.read_csv('dados_tratados.csv') 

# DataFrame de avaliações
df_avaliacoes = pd.read_csv('avaliacoes_simuladas.csv')

# matriz de similaridade
cosine_sim_matrix = np.load('cosine_similarity_matrix.npy')

# Um mapeamento do ID do artigo para o índice da linha no DataFrame
# encontrar rapidamente a posição de um artigo na matriz
indices = pd.Series(df_artigos.index, index=df_artigos['ids']).drop_duplicates()


# FUNÇÃO PRINCIPAL DE RECOMENDAÇÃO

def obter_recomendacoes(user_id, top_n=10):
    """
    Gera uma lista de recomendações para um usuário específico,
    baseado no conteúdo dos artigos que ele já avaliou positivamente
    """
    
    # Encontrar artigos que usuário avaliou
    avaliacoes_do_usuario = df_avaliacoes[df_avaliacoes['user_id'] == user_id]
    
    # Filtrar para pegar apenas artigos que usuário gostou (nota >= 4)
    artigos_que_gostou = avaliacoes_do_usuario[avaliacoes_do_usuario['rating'] >= 4]
    
    if artigos_que_gostou.empty:
        return ["Não há avaliações positivas suficientes para gerar recomendações."]

    # Para cada artigo que usuário gostou, acumular scores de similaridade
    scores_agregados = {}
    for index, row in artigos_que_gostou.iterrows():
        artigo_id = row['artigo_id']
        nota = row['rating']
        
        # Pega índice do artigo na matriz de similaridade
        idx_artigo = indices[artigo_id]
        
        # Pega a linha de scores de similaridade para este artigo
        # e pondera pela nota (um item nota 5 tem mais influência)
        vetor_similaridade = cosine_sim_matrix[idx_artigo] * nota

        # Soma os scores ponderados no dicionário agregado
        for idx_similar, score in enumerate(vetor_similaridade):
            scores_agregados[idx_similar] = scores_agregados.get(idx_similar, 0) + score

    # Remover artigos que usuário já avaliou da lista de candidatos
    indices_ja_avaliados = [indices[artigo_id] for artigo_id in avaliacoes_do_usuario['artigo_id']]
    for idx in indices_ja_avaliados:
        if idx in scores_agregados:
            del scores_agregados[idx]

    # Ordenar artigos restantes pelo score agregado e pegar os 'top_n' melhores
    indices_recomendados = sorted(scores_agregados, key=scores_agregados.get, reverse=True)[:top_n]
    
    # Retornar os títulos dos artigos recomendados
    return df_artigos.iloc[indices_recomendados]['titles'].tolist()


# TESTANDO A FUNÇÃO
print("Função 'obter_recomendacoes' definida. Vamos testá-la...")

# Pega o primeiro ID de usuário único do dataset de avaliações para teste
id_usuario_teste = df_avaliacoes['user_id'].unique()[0]

print(f"\\n--- TESTE PARA O USUÁRIO: {id_usuario_teste} ---")

# Mostra o que esse usuário gostou pra podermos validar recomendação
avaliacoes_positivas_teste = df_avaliacoes[(df_avaliacoes['user_id'] == id_usuario_teste) & (df_avaliacoes['rating'] >= 4)]
titulos_gostou = df_artigos[df_artigos['ids'].isin(avaliacoes_positivas_teste['artigo_id'])]['titles'].tolist()

print("\\nEste usuário gostou de:")
for titulo in titulos_gostou:
    print(f"- {titulo}")

# Chama função para gerar recomendações
recomendacoes = obter_recomendacoes(user_id=id_usuario_teste, top_n=7)

print("\\nRecomendações geradas pelo sistema:")
for i, rec_titulo in enumerate(recomendacoes):
    print(f"{i+1}. {rec_titulo}")



# Bibliotecas necessárias pra interface
import ipywidgets as widgets
from IPython.display import display, clear_output
import pandas as pd

# WIDGETS DA INTERFACE

# caixa de texto pro nome do novo usuário
input_usuario = widgets.Text(
    placeholder='Digite seu nome aqui...',
    description='Novo Usuário:',
    disabled=False
)

# Botão para iniciar o processo
botao_cadastrar = widgets.Button(
    description='Criar Perfil',
    button_style='success', # 'success', 'info', 'warning', 'danger' or ''
    tooltip='Clique para iniciar a avaliação de artigos e criar seu perfil.',
    icon='user-plus'
)

# Container para os widgets de avaliação que serão criados dinamicamente
box_perfil_avaliacoes = widgets.VBox([])

# Container para a saída final das recomendações
output_final = widgets.Output()


# LÓGICA DA INTERFACE

def iniciar_criacao_perfil(b):
    """
    Esta função é chamada quando o botão 'Criar Perfil' é clicado.
    Ela monta a tela de avaliação de artigos.
    """
    # Pega o nome do usuário digitado
    nome_usuario = input_usuario.value
    if not nome_usuario:
        print("Por favor, digite um nome de usuário antes de continuar.")
        return

    # Limpa a tela anterior
    clear_output(wait=True)
    
    # Seleciona 10 artigos aleatórios para o usuário avaliar
    artigos_para_avaliar = df_artigos.sample(10)
    
    # Cria os componentes da tela de avaliação
    label_perfil = widgets.Label("Para entendermos seu gosto, avalie os seguintes artigos de 1 a 5:")
    
    # Cria um slider para cada artigo a ser avaliado
    sliders = []
    for _, row in artigos_para_avaliar.iterrows():
        slider = widgets.IntSlider(
            min=1, max=10, value=3, # Começa com nota 3 (neutra)
            description=row['titles'], 
            style={'description_width': 'initial'} # Evita que o título seja cortado
        )
        # Anexa o ID do artigo ao widget para uso posterior
        slider.artigo_id = row['ids'] 
        sliders.append(slider)
        
    botao_recomendar = widgets.Button(description='Gerar Recomendações', button_style='info', icon='lightbulb')
    
    # Define o que acontece quando o botão 'Gerar Recomendações' é clicado
    def obter_recomendacoes_novo_usuario(b):
        # Limpa a tela de avaliação
        clear_output(wait=True)
        # Mostra a área de saída final
        display(output_final)
        
        with output_final:
            print(f"Olá, {nome_usuario}! Com base no seu perfil, aqui estão suas recomendações:")
            
            # Coleta as avaliações dos sliders
            novas_avaliacoes = [{'user_id': nome_usuario, 'artigo_id': s.artigo_id, 'rating': s.value} for s in sliders]
            novas_avaliacoes_df = pd.DataFrame(novas_avaliacoes)
            
            # Adiciona as novas avaliações ao DataFrame global de avaliações
            global df_avaliacoes
            df_avaliacoes = pd.concat([df_avaliacoes, novas_avaliacoes_df], ignore_index=True)
            
            # Chama a sua função principal para obter as recomendações
            recomendacoes = obter_recomendacoes(user_id=nome_usuario, top_n=5)
            
            # Exibe as recomendações de forma legível
            print("-" * 50)
            for i, titulo in enumerate(recomendacoes):
                print(f"{i+1}. {titulo}")
            print("-" * 50)

    botao_recomendar.on_click(obter_recomendacoes_novo_usuario)
    
    # Monta e exibe a tela de avaliação
    box_perfil_avaliacoes.children = [label_perfil] + sliders + [botao_recomendar]
    display(box_perfil_avaliacoes)

# Conecta a função ao clique do botão de cadastro inicial
botao_cadastrar.on_click(iniciar_criacao_perfil)

Função 'obter_recomendacoes' definida. Vamos testá-la...
\n--- TESTE PARA O USUÁRIO: 420 ---
\nEste usuário gostou de:
- View-Consistent 3D Editing with Gaussian Splatting
- CAT: Causal Audio Transformer for Audio Classification
- Deep Attention Fusion Feature for Speech Separation with End-to-End
  Post-filter Method
- Towards Modeling Human Motor Learning Dynamics in High-Dimensional
  Spaces
- HPCGen: Hierarchical K-Means Clustering and Level Based Principal
  Components for Scan Path Genaration
- Hybrid Base Complex: Extract and Visualize Structure of Hex-dominant
  Meshes
- Computing complete hyperbolic structures on cusped 3-manifolds
- Alchymical Mirror: Real-time Interactive Sound- and Simple
  Motion-Tracking Set of Jitter/Max/MSP Patches
- Optimally Guarding 2-Reflex Orthogonal Polyhedra by Reflex Edge Guards
- Lana: A Language-Capable Navigator for Instruction Following and
  Generation
- Generative Novel View Synthesis with 3D-Aware Diffusion Models
- Security, Availability

In [83]:
# Conta quantas avaliações positivas (>=4) cada artigo recebeu
popularidade = df_avaliacoes[df_avaliacoes['rating'] >= 4]['artigo_id'].value_counts()

# Converte para um DataFrame e junta com os títulos
df_populares = pd.DataFrame({'ids': popularidade.index, 'contagem_positiva': popularidade.values})
df_populares = pd.merge(df_populares, df_artigos[['ids', 'titles']], on='ids')
df_populares = df_populares.sort_values(by='contagem_positiva', ascending=False)

print("Artigos mais populares:")
display(df_populares.head())

Artigos mais populares:


,ids,contagem_positiva,titles
0,2404.12498,4,A Configurable Pythonic Data Center Model for ...
1,2205.07722,4,How Different Groups Prioritize Ethical Values...
2,2309.08051,3,Retrieval-Augmented Text-to-Audio Generation
3,2207.14139,3,Nominal Matching Logic
4,2309.14034,3,A unified worst case for classical simplex and...


In [ ]:
import pandas as pd
import numpy as np
import ipywidgets as widgets
from IPython.display import display, clear_output

# Carregando todos os componentes necessários 

# DataFrame de artigos
df_artigos = pd.read_csv('dados_tratados.csv') 

# DataFrame de avaliações
df_avaliacoes = pd.read_csv('avaliacoes_simuladas.csv')

# Matriz de similaridade
cosine_sim_matrix = np.load('cosine_similarity_matrix.npy')

# 4. Mapeamento do ID do artigo para o índice da linha no DataFrame
indices = pd.Series(df_artigos.index, index=df_artigos['ids']).drop_duplicates()

# Cálculo da Popularidade dos Artigos
print("Calculando a popularidade dos artigos...")
popularidade = df_avaliacoes[df_avaliacoes['rating'] >= 4]['artigo_id'].value_counts()
df_populares = pd.DataFrame({'ids': popularidade.index, 'contagem_positiva': popularidade.values})
df_populares = pd.merge(df_populares, df_artigos[['ids', 'titles']], on='ids')
df_populares = df_populares.sort_values(by='contagem_positiva', ascending=False)
print("Cálculo de popularidade concluído.")


# FUNÇÃO PRINCIPAL DE RECOMENDAÇÃO 
def obter_recomendacoes(user_id, top_n=10):
    avaliacoes_do_usuario = df_avaliacoes[df_avaliacoes['user_id'] == user_id]
    
    # Tenta pegar artigos com nota alta (>= 4)
    artigos_que_gostou = avaliacoes_do_usuario[avaliacoes_do_usuario['rating'] >= 4]
    
    # Se não houver artigos com nota alta, tenta usar os de nota média (3)
    if artigos_que_gostou.empty:
        print("\n>> (Info: Nenhuma avaliação positiva encontrada. Tentando com avaliações neutras...)")
        artigos_que_gostou = avaliacoes_do_usuario[avaliacoes_do_usuario['rating'] == 3]
        
        # Se ainda assim não houver nenhum, retorna lista vazia
        if artigos_que_gostou.empty:
            return []
    
    scores_agregados = {}
    for index, row in artigos_que_gostou.iterrows():
        artigo_id = row['artigo_id']
        nota = row['rating']
        
        if artigo_id not in indices:
            continue
            
        idx_artigo = indices[artigo_id]
        vetor_similaridade = cosine_sim_matrix[idx_artigo] * nota

        for idx_similar, score in enumerate(vetor_similaridade):
            scores_agregados[idx_similar] = scores_agregados.get(idx_similar, 0) + score

    indices_ja_avaliados = [indices[artigo_id] for artigo_id in avaliacoes_do_usuario['artigo_id'] if artigo_id in indices]
    for idx in indices_ja_avaliados:
        if idx in scores_agregados:
            del scores_agregados[idx]

    indices_recomendados = sorted(scores_agregados, key=scores_agregados.get, reverse=True)[:top_n]
    
    return df_artigos.iloc[indices_recomendados]['titles'].tolist()

# FUNÇÃO ROBUSTA COM FALLBACK
def obter_recomendacoes_robusta(user_id, top_n=10):
    """
    Tenta obter recomendações por conteúdo (usando notas altas e depois médias). 
    Se falhar, retorna os artigos mais populares que o usuário ainda não viu.
    """
    recomendacoes = obter_recomendacoes(user_id, top_n)
    
    if not recomendacoes:
        print("\n>> Não encontramos recomendações personalizadas para o seu perfil.")
        print(">> Sugerindo os artigos mais populares que você ainda não viu:")
        
        artigos_ja_vistos = df_avaliacoes[df_avaliacoes['user_id'] == user_id]['artigo_id'].tolist()
        
        recomendacoes_populares = df_populares[~df_populares['ids'].isin(artigos_ja_vistos)]
        
        return recomendacoes_populares.head(top_n)['titles'].tolist()
    
    print("\n>> Recomendações personalizadas geradas com sucesso:")
    return recomendacoes


# LÓGICA DA INTERFACE
import ipywidgets as widgets
from IPython.display import display, clear_output

input_usuario = widgets.Text(placeholder='Digite seu nome aqui...', description='Novo Usuário:')
botao_cadastrar = widgets.Button(description='Criar Perfil', button_style='success', tooltip='Clique para criar seu perfil.', icon='user-plus')
box_perfil_avaliacoes = widgets.VBox([])
output_final = widgets.Output()

def iniciar_criacao_perfil(b):
    nome_usuario = input_usuario.value
    if not nome_usuario:
        print("Por favor, digite um nome de usuário antes de continuar.")
        return

    clear_output(wait=True)
    
    artigos_para_avaliar = df_artigos.sample(7)
    
    label_perfil = widgets.Label("Para entendermos seu gosto, avalie os seguintes artigos de 1 a 5:")
    
    sliders = []
    for _, row in artigos_para_avaliar.iterrows():
        # slider para a nota
        slider = widgets.IntSlider(
            min=1, max=5, value=3,
            step=1,
            description=row['titles'], 
            style={'description_width': 'initial'}, # Garante que o título não seja cortado
            layout=widgets.Layout(width='95%') # Ocupa a maior parte do espaço
        )
        # Anexa o ID do artigo ao widget pra usar depois
        slider.artigo_id = row['ids'] 
        sliders.append(slider)
        
    botao_recomendar = widgets.Button(description='Gerar Recomendações', button_style='info', icon='lightbulb')
    
    def obter_recomendacoes_novo_usuario(b):
        clear_output(wait=True)
        display(output_final)
        
        with output_final:
            print(f"Olá, {nome_usuario}! Processando seu perfil...")
            
            novas_avaliacoes = [{'user_id': nome_usuario, 'artigo_id': s.artigo_id, 'rating': s.value} for s in sliders]
            novas_avaliacoes_df = pd.DataFrame(novas_avaliacoes)
            
            global df_avaliacoes
            df_avaliacoes = pd.concat([df_avaliacoes, novas_avaliacoes_df], ignore_index=True)
            
            recomendacoes = obter_recomendacoes_robusta(user_id=nome_usuario, top_n=5)
            
            print("-" * 50)
            for i, titulo in enumerate(recomendacoes):
                print(f"{i+1}. {titulo}")
            print("-" * 50)

    botao_recomendar.on_click(obter_recomendacoes_novo_usuario)
    
    box_perfil_avaliacoes.children = [label_perfil] + sliders + [botao_recomendar]
    display(box_perfil_avaliacoes)

botao_cadastrar.on_click(iniciar_criacao_perfil)


# EXIBIÇÃO INICIAL DA INTERFACE
print("\nSistema de recomendação pronto.")
display(input_usuario, botao_cadastrar)

Output()